<a href="https://colab.research.google.com/github/SaraEstilaei/incident-sentiment-analyzer/blob/Model%2FNLP_03_nlptown/nlp_03_nlptown_final_with__ans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # فقط یک GPU (با دو GPU، Trainer خودکار batch را دو برابر می‌کند)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, glob, inspect
import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ندارد")

GPU: Tesla T4


<div dir="rtl">

# مدل ۲: nlptown/bert-base-multilingual-uncased-sentiment

- همان split مدل DeBERTa استفاده شده (۵۶k/۱۶k/۸k) و هم‌پوشانی صفر با `assert` چک شده.
- این مدل از قبل head پنج‌کلاسه داره (۱ تا ۵ ستاره)، با `num_labels=5` همون head حفظ میشه.
- **دلیل انتخاب:** خانواده و tokenizer متفاوت از DeBERTa، یعنی خطاهای متفاوت و مناسب برای ensemble.
- فقط ۲ epoch آموزش داده شد؛ بهترین micro-F1 در epoch دوم (۰٫۷۲۴۳) بود.

</div>

In [ ]:
SEED       = 42
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
RUN_NAME   = "nlptown_80k"
MAX_LEN    = 128
EPOCHS  = 2 # طبق تجربه‌ی DeBERTa، همین 1 یا 2 اپوک کافیه
LR         = 2e-5
BS         = 32

SAVE_ROOT = f"/kaggle/working/{RUN_NAME}"
CKPT_DIR  = f"{SAVE_ROOT}/checkpoints"

os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

set_seed(SEED)
assert torch.cuda.is_available(), "GPU روشن نیست: Settings > Accelerator > GPU T4"

In [ ]:
def find_csv(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(hits) == 1, f"{name}: {len(hits)} فایل پیدا شد: {hits}"
    return hits[0]

TRAIN_PATH, VAL_PATH, TEST_PATH = (find_csv(n) for n in ["train.csv", "val.csv", "test.csv"])
print(TRAIN_PATH, VAL_PATH, TEST_PATH, sep="\n")

/kaggle/input/datasets/kianaghaznavi/nlp-03-nlptown-final-dataset/train.csv
/kaggle/input/datasets/kianaghaznavi/nlp-03-nlptown-final-dataset/val.csv
/kaggle/input/datasets/kianaghaznavi/nlp-03-nlptown-final-dataset/test.csv


In [ ]:
train_df, val_df, test_df = (pd.read_csv(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH])
print("train / val / test:", train_df.shape, val_df.shape, test_df.shape, "(انتظار: 56000 / 16000 / 8000)")

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert {"full_text", "label"} <= set(d.columns), f"{name}: ستون‌ها ناقص: {d.columns.tolist()}"
    assert d["full_text"].notna().all() and d["label"].notna().all()
    d["label"] = d["label"].astype(int)
    assert set(d["label"].unique()) <= set(range(5)), "label باید بین 0 تا 4 باشد"
    print(name, d["label"].value_counts(normalize=True).sort_index().round(3).to_dict())

# بدون نشت: هیچ متنی نباید هم‌زمان در دو مجموعه باشد
tr, va, te = (set(d["full_text"]) for d in (train_df, val_df, test_df))
print("هم‌پوشانی train-val / train-test / val-test:", len(tr & va), len(tr & te), len(va & te))
assert not (tr & va) and not (tr & te) and not (va & te), "نشت داده!"

train / val / test: (56000, 4) (16000, 4) (8000, 4) (انتظار: 56000 / 16000 / 8000)
train {0: 0.099, 1: 0.068, 2: 0.097, 3: 0.187, 4: 0.55}
val {0: 0.099, 1: 0.068, 2: 0.097, 3: 0.187, 4: 0.55}
test {0: 0.099, 1: 0.068, 2: 0.097, 3: 0.187, 4: 0.55}
هم‌پوشانی train-val / train-test / val-test: 0 0 0


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(texts, labels):
    ds = Dataset.from_dict({"text": list(texts), "label": [int(x) for x in labels]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds = make_ds(train_df["full_text"], train_df["label"])
val_ds   = make_ds(val_df["full_text"],   val_df["label"])

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"micro_f1": f1_score(p.label_ids, preds, average="micro"),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/56000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [ ]:
_ta = inspect.signature(TrainingArguments.__init__).parameters
EVAL_KW   = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
WARMUP_KW = "warmup_ratio"  if "warmup_ratio"  in _ta else "warmup_steps"
TOK_KW    = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"


# این مدل از قبل یک هد پنج‌کلاسه (۱ تا ۵ ستاره) دارد؛ با num_labels=5 همون هد حفظ میشه
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)
print("id2label مدل:", model.config.id2label)    # فقط برای اطلاع؛ fine-tune با label های خودمان انجام می‌شود

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    weight_decay=0.01,
    **{WARMUP_KW: 0.06},
    **{EVAL_KW: "epoch"},
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    save_total_limit=2,
    logging_steps=200,
    fp16=True,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    **{TOK_KW: tok},
)

has_ckpt = len(glob.glob(f"{CKPT_DIR}/checkpoint-*")) > 0
print("ادامه از چک‌پوینت:", has_ckpt)
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id2label مدل: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}
ادامه از چک‌پوینت: False


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.701428,0.699300,0.721250,0.566532
2,0.603255,0.705441,0.724313,0.586079


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3500, training_loss=0.677891746520996, metrics={'train_runtime': 792.777, 'train_samples_per_second': 141.276, 'train_steps_per_second': 4.415, 'total_flos': 7367307988992000.0, 'train_loss': 0.677891746520996, 'epoch': 2.0})

In [ ]:
def predict_probs(texts, bs=128):
    """احتمال softmax هر ۵ کلاس؛ ترتیب خروجی دقیقاً برابر ترتیب ورودی است."""
    m = trainer.model.eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])                # مرتب‌سازی بر اساس طول (فقط برای سرعت)
    out = np.zeros((len(texts), 5), dtype=np.float32)
    for i in range(0, len(texts), bs):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(m.device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            out[idx] = m(**enc).logits.float().cpu().numpy()   # هر نتیجه به جای اصلی خودش برمی‌گردد
    return softmax(out, axis=1)

def evaluate(name, d):
    probs = predict_probs(d["full_text"])
    y, pred = d["label"].values, probs.argmax(1)
    micro = f1_score(y, pred, average="micro")
    macro = f1_score(y, pred, average="macro")
    ll    = log_loss(y, probs, labels=range(5))
    se    = 1.96 * np.sqrt(micro * (1 - micro) / len(y))
    print(f"\n===== {name} ({len(y)} نمونه) =====")
    print(f"micro-F1: {micro:.4f} (±{se:.3f}) | macro-F1: {macro:.4f} | log-loss: {ll:.4f}")
    print(classification_report(y, pred, target_names=["1", "2", "3", "4", "5"], zero_division=0))
    print("Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):")
    print(confusion_matrix(y, pred))
    return probs

probs_val  = evaluate("validation", val_df)
probs_test = evaluate("test", test_df)

# ذخیره (مدل برای تحویل؛ فایل‌های npy برای ensemble با DeBERTa)
trainer.save_model(f"{SAVE_ROOT}/model")
tok.save_pretrained(f"{SAVE_ROOT}/model")
np.save(f"{SAVE_ROOT}/val_probs.npy",   probs_val)
np.save(f"{SAVE_ROOT}/test_probs.npy",  probs_test)
np.save(f"{SAVE_ROOT}/val_labels.npy",  val_df["label"].values)
np.save(f"{SAVE_ROOT}/test_labels.npy", test_df["label"].values)

print("\nذخیره شد:")
for root, _, files in os.walk(SAVE_ROOT):
    for f in files:
        p = os.path.join(root, f)
        print(f"{p}  ({os.path.getsize(p) / 1e6:.1f} MB)")


===== validation (16000 نمونه) =====
micro-F1: 0.7243 (±0.007) | macro-F1: 0.5861 | log-loss: 0.7054
              precision    recall  f1-score   support

           1       0.68      0.72      0.70      1578
           2       0.43      0.33      0.37      1084
           3       0.47      0.50      0.48      1551
           4       0.56      0.46      0.50      2994
           5       0.85      0.90      0.88      8793

    accuracy                           0.72     16000
   macro avg       0.60      0.58      0.59     16000
weighted avg       0.71      0.72      0.72     16000

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[1133  220  113   38   74]
 [ 309  353  297   69   56]
 [ 121  186  771  348  125]
 [  50   37  353 1385 1169]
 [  60   24  110  652 7947]]

===== test (8000 نمونه) =====
micro-F1: 0.7206 (±0.010) | macro-F1: 0.5810 | log-loss: 0.7115
              precision    recall  f1-score   support

           1       0.68      0.71      0.69       789
           2     

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ذخیره شد:
/kaggle/working/nlptown_80k/test_probs.npy  (0.2 MB)
/kaggle/working/nlptown_80k/val_probs.npy  (0.3 MB)
/kaggle/working/nlptown_80k/test_labels.npy  (0.1 MB)
/kaggle/working/nlptown_80k/val_labels.npy  (0.1 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/scheduler.pt  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/scaler.pt  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/model.safetensors  (669.5 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/config.json  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/rng_state.pth  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/trainer_state.json  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/training_args.bin  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/tokenizer_config.json  (0.0 MB)
/kaggle/working/nlptown_80k/checkpoints/checkpoint-3500/tokenizer.json  (2.6 MB)
/kaggle/working/nlptown_80k/checkpoints/

In [ ]:
import shutil

shutil.rmtree(
    "/kaggle/working/nlptown_80k/checkpoints"
)
shutil.make_archive(
    "/kaggle/working/nlptown_80k",
    "zip",
    "/kaggle/working/nlptown_80k"
)

'/kaggle/working/nlptown_80k.zip'

In [ ]:
MODEL_PATH = "/kaggle/input/NLP_03_nlptown_final/model"

In [ ]:
import os

print("Datasets available in Kaggle:\n")

for name in os.listdir("/kaggle/input"):
    print(name)
    path = os.path.join("/kaggle/input", name)

    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            level = root.replace(path, "").count(os.sep)
            if level > 2:
                dirs[:] = []
                continue

            print("  " * (level + 1) + os.path.basename(root) + "/")

            for file in files:
                print("  " * (level + 2) + file)

Datasets available in Kaggle:

notebooks
  notebooks/
    kianaghaznavi/
      nlp-03-nlptown-final/
        __huggingface_repos__.json
        __notebook__.ipynb
        __output__.json
        __results__.html
        custom.css
        nlptown_80k.zip
datasets
  datasets/
    kianaghaznavi/
      nlp-03-nlptown-final-dataset/
        test.csv
        test_cleaned.csv
        train.csv
        val.csv


In [ ]:
import zipfile
import os

ZIP_PATH = "/kaggle/input/notebooks/kianaghaznavi/nlp-03-nlptown-final/nlptown_80k.zip"
EXTRACT_PATH = "/kaggle/working/nlptown_model"

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

MODEL_PATH = "/kaggle/working/nlptown_model/model"
print(os.listdir(MODEL_PATH))

['model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'config.json']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

TEST_PATH = "/kaggle/input/datasets/kianaghaznavi/nlp-03-nlptown-final-dataset/test_cleaned.csv"

test_df = pd.read_csv(TEST_PATH)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
display(test_df.head(2))

Shape: (20000, 13)
Columns: ['vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'review_year', 'reviewText_clean', 'full_text']


,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,review_year,reviewText_clean,full_text
0,0,1,2018-06-04,A20GGWE66JW9X2,B006Z394GM,{'Color:': ' FPS01-C'},Brian C Toner,The name and description of this device are mi...,The prize for most useless invention of all ti...,1528070400,2018,The name and description of this device are mi...,The prize for most useless invention of all ti...
1,0,1,2018-02-18,ARYJWXHEYHG9M,B005BE058W,"{'Size:': ' 1000W', 'Style:': ' G2'}",Snake,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR,1518912000,2018,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR One of the molex connec...


In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

# Input texts
texts = test_df["full_text"].fillna("").astype(str).tolist()

# Prediction
def predict_probs_final(texts, batch_size=128, max_length=128):

    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = tok(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)

        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)


final_probs = predict_probs_final(texts)

print("Prediction shape:", final_probs.shape)
print("Probability sum:", final_probs[0].sum())

  0%|          | 0/157 [00:00<?, ?it/s]

Prediction shape: (20000, 5)
Probability sum: 0.99999994


In [ ]:
np.save(
    "/kaggle/working/nlptown_final_probs.npy",
    final_probs
)

print("Saved successfully!")

Saved successfully!
